# Where is an after-cost opportunity plausible?

**P0 · A1–A6.** Decide where to spend research effort before choosing a signal or model.
Owner: researcher. Design refreshed 2026-09-23; no empirical study has been run here.

This workbook specifies an opportunity map and displays **supplied daily measurements**.
It does not infer a trade from an unconditional price move. Start with the
[agenda](research-agenda.md) and retain a [study protocol](study-protocol.md).

## 1. Narrow the mandate

Select one execution venue, one or two instruments, and spot or a specified perpetual
contract. Distinguish base quantity, quote currency, contract multiplier and price tick.
BTC and ETH are candidates, not an automatic universe. A second venue starts as information.

Compare a small fixed horizon grid such as 250 ms, 1 s, 5 s, 30 s and 120 s. Measure
fast horizons without assuming the stack can act on them. Cross horizon with order size,
execution style and **measured or explicitly stressed** delay. Record observation,
processing, submission and cancellation delay separately in the retained study.

**Question:** is the accessible movement scale plausibly large relative to the complete
trading cost, at useful size and frequency? **Competing explanation:** apparently large
moves occur in stale, thin, gapped or untradeable states.

## 2. Establish the sample before measuring opportunity

Use multiple days and liquidity/volatility conditions. Retain the exact product,
channel, feed/fee regimes, historical eligibility and calendar coverage. Count requested,
valid and excluded time; an empty recording does not establish a quiet market.

Inspect tick and spread distributions, depth at fixed quote distances, update cadence,
resynchronizations and source/receipt clock uncertainty. A snapshot/update feed does not
necessarily expose every exchange event. Aggregate L2 does not identify queue priority.

Remove invalid/stale windows under a declared rule and report the removed share. Compare
results with and without suspect episodes. Full-day validity and independent-day coverage
remain research questions, even when replay reproduces the saved bytes.

## 3. Measure movement and the cost hurdle separately

For each valid decision time, measure future midprice movement as a **diagnostic** and
entry/exit prices under the specified execution scenario. Use fixed times or a past-only
event trigger. Never choose entry direction from the future move or select only daily maxima.

For a long completed round trip: `net = quantity × (sell_price − buy_price) − fees − financing`.
Executable prices already include spread and depth slippage; do not subtract them again.
Include both legs, depth participation, failed exits and residual inventory. Historical fees
and a hypothetical current-account fee scenario are separate experiments.

For passive scenarios, leave costs unestimated until a plausible joint fill/markout model
exists. A spread-capture assumption is not a passive cost estimate. Record capacity against
capital and operating costs separately from statistical significance.

## 4. Build the daily opportunity map

Populate `measurements` from retained exploratory-study outputs. Each row represents
one day and one complete scenario; every numeric value must be measured or explicitly
modeled in the referenced record. Do not use this table to inspect a final holdout.

`valid_seconds + excluded_seconds` covers the **requested session**, not necessarily
24 hours. `opportunities` counts predeclared eligible decisions. Movement fields are
absolute midprice moves in basis points; cost fields are all-in round-trip estimates
in basis points of entry notional. Quantiles are computed within each day/scenario.
No synthetic rows are supplied. Empty input leaves the study awaiting evidence.

In [ ]:
# standard library imports
from math import isfinite

# third party library imports
import pandas as pd

measurements: list[dict[str, object]] = []
axes = [
    "venue",
    "instrument",
    "product",
    "quote_currency",
    "day",
    "horizon_ms",
    "execution_style",
    "size_base",
    "delay_scenario",
]
metrics = [
    "valid_seconds",
    "excluded_seconds",
    "opportunities",
    "spread_p50_bps",
    "spread_p90_bps",
    "depth_p10_quote",
    "abs_move_p50_bps",
    "abs_move_p90_bps",
    "roundtrip_cost_p50_bps",
    "roundtrip_cost_p90_bps",
]
opportunity_map = pd.DataFrame(measurements, columns=[*axes, *metrics, "evidence_ref"])

In [ ]:
if opportunity_map.empty:
    print("AWAITING EVIDENCE — populate measured daily scenarios; no opportunity is established.")
else:
    if opportunity_map.isna().any().any() or opportunity_map.duplicated(axes).any():
        raise ValueError("Each complete day/scenario requires one non-missing row.")
    for field in [*metrics, "horizon_ms", "size_base"]:
        opportunity_map[field] = pd.to_numeric(opportunity_map[field], errors="raise")
    if not opportunity_map[[*metrics, "horizon_ms", "size_base"]].map(isfinite).all().all():
        raise ValueError("Measurements must be finite.")
    if (opportunity_map[["valid_seconds", "excluded_seconds", "opportunities"]] < 0).any().any():
        raise ValueError("Coverage and opportunity counts cannot be negative.")
    requested_seconds = opportunity_map.valid_seconds + opportunity_map.excluded_seconds
    if (requested_seconds <= 0).any() or (
        opportunity_map[["horizon_ms", "size_base"]] <= 0
    ).any().any():
        raise ValueError("Requested duration, horizon and size must be positive.")
    nonnegative = [field for field in metrics if not field.startswith("roundtrip_cost")]
    if (opportunity_map[nonnegative] < 0).any().any():
        raise ValueError("Spread, depth, absolute movement and coverage cannot be negative.")
    for prefix in ("spread", "abs_move", "roundtrip_cost"):
        if (opportunity_map[f"{prefix}_p50_bps"] > opportunity_map[f"{prefix}_p90_bps"]).any():
            raise ValueError("Within-day p50 must not exceed p90.")
    if (opportunity_map.opportunities % 1 != 0).any():
        raise ValueError("Opportunity counts must be integers.")
    if opportunity_map.evidence_ref.astype(str).str.strip().eq("").any():
        raise ValueError("Every scenario needs a retained evidence reference.")
    opportunity_map["valid_fraction"] = opportunity_map.valid_seconds / requested_seconds
    display(opportunity_map.sort_values(axes))

## 5. Interpret coverage, delay and size together

Inspect each daily row rather than averaging daily quantiles into a supposed pooled
quantile. Compare the same sample under delay and size stresses. Count days and market
conditions, not book updates, when describing the strength of evidence.

Movement larger than the hurdle means only that a directionally informed policy could
be worth testing. It does not say that direction is observable in advance. Ask whether
candidate events are frequent enough, whether costs change in those events, and whether
removing invalid periods removes the apparent opportunity.

**Continue:** choose a bounded venue/instrument/horizon region and a precise hypothesis.
**Narrow:** restrict size, horizon or state where the mechanism remains plausible.
**Reject:** feasible delays, costs or capacity make the opportunity irrelevant.
**Collect more:** coverage or economic inputs cannot support a decision.

## 6. Record the decision

Complete the [decision memo](study-protocol.md#decision-memo): chosen opportunity set,
coverage, source and fee evidence, scenario sensitivities, missing inputs, and next question.
The next study is [selective liquidity-taking](liquidity-taking.ipynb). Test pressure and
recent-return baselines on common data; do not optimize a signal while constructing this map.

**Current conclusion: insufficient evidence.** An empty or successfully executed worksheet
is not a completed opportunity map.